# CENG 467 — Evaluate all systems (ROUGE + BERTScore + error analysis)
Run after training. Generates predictions for every system and produces the master metrics table.


## 1. Mount + setup (same as train notebook)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
os.chdir('/content/drive/MyDrive/ceng467_termproject/CENG467-term-project')
!pip install -q -r requirements.txt


## 2. Load API keys


In [ ]:
import os
os.environ['OPENAI_API_KEY']    = os.environ.get('OPENAI_API_KEY')    or 'sk-...'
os.environ['ANTHROPIC_API_KEY'] = os.environ.get('ANTHROPIC_API_KEY') or 'sk-ant-...'


## 3. Run the evaluation script


In [ ]:
!bash scripts/04_evaluate.sh


## 4. Inspect aggregated metrics


In [ ]:
import json, pandas as pd
with open('outputs/results/main.json') as f:
    data = json.load(f)
rows = []
for label, sys in data.items():
    row = {'system': label, 'n': sys['n']}
    if 'rouge_standard' in sys:
        for k in ('rouge1', 'rouge2', 'rougeL'):
            row[f'{k}_std'] = sys['rouge_standard'][k]
    if 'rouge_stem' in sys:
        for k in ('rouge1', 'rouge2', 'rougeL'):
            row[f'{k}_stem'] = sys['rouge_stem'][k]
    if 'bertscore' in sys:
        row['bertscore_f1'] = sys['bertscore']['f1']
    if 'errors' in sys:
        row.update({k: sys['errors'][k] for k in sys['errors'] if k != 'n'})
    rows.append(row)
pd.DataFrame(rows).set_index('system').round(4)


## 5. OOD evaluation on TR-News


In [ ]:
!python -m src.student.infer \
  --model-path outputs/checkpoints/openai_concise_n10000_r8/final \
  --input data/raw/trnews/test.jsonl \
  --out outputs/predictions/ood_S_gpt.jsonl
!python -m src.eval.run_eval \
  --pred S-gpt=outputs/predictions/ood_S_gpt.jsonl \
  --out-json outputs/results/ood.json
